In [2]:
# Cell 0 — Overview & configuration (xG feature plan)

FEATURES_DOC = """
This notebook assembles per-shot features for a handball xG model.

Timestamp priority (release):
1) main.kinexon_throwpoint_refined.refined_throw_ts
2) main.kinexon_events_detected_enriched.timestamp_ms (or detected_shot_handball in vw)
3) match_events.eventTime (rare fallback)

Feature families:
- Geometry (at release):
  * shot_x, shot_y
  * dist_to_goal_center (m)
  * open_angle_to_goal (rad)
  * bearing_to_goal_center (rad)
- Kinematics (at release):
  * shooter_speed (m/s), shooter_acc (m/s²)
  * ball_speed (m/s), ball_acc (m/s²) if aligned
  * runup_angle_diff_to_goal (rad)
- Goalkeeper (if GK id known at release):
  * gk_x, gk_y, gk_dist_to_center, gk_lateral_offset, gk_line_depth
  * shooter_to_gk_distance (m)
- Defensive pressure (instant):
  * opp_within_1_5m, opp_within_3m
  * opp_in_shooting_cone_[15,30]deg (counts)
- Context:
  * is_goal (label)
  * home/away indicator for shooter team (if derivable)
  * game_clock/period if available (optional)

Notes:
- Field coords assumed centered at (0,0), x-axis from attacking left to right; posts at (GOAL_X, ±GOAL_HALF_WIDTH).
- If team direction is unknown, geometry uses the *nearest goal* from the shooter's x (robust for both halves).
"""
import numpy as np
import pandas as pd
from IPython.display import display
print(FEATURES_DOC.strip())

# Geometry constants (tune if needed)
GOAL_X_ABS = 20.0           # |x| coordinate of goal line (m) in Kinexon pitch space (~40m width)
GOAL_HALF_WIDTH = 1.5       # goal half width (m); width ≈ 3.0 m
PRESSURE_R1 = 1.5           # tight pressure radius (m)
PRESSURE_R2 = 3.0           # loose pressure radius (m)
CONE_15 = np.deg2rad(15.0)  # narrow cone toward goal
CONE_30 = np.deg2rad(30.0)  # wider cone toward goal
NEAREST_TS_TOL_MS = 30      # asof tolerance for joining tracks at release


This notebook assembles per-shot features for a handball xG model.

Timestamp priority (release):
1) main.kinexon_throwpoint_refined.refined_throw_ts
2) main.kinexon_events_detected_enriched.timestamp_ms (or detected_shot_handball in vw)
3) match_events.eventTime (rare fallback)

Feature families:
- Geometry (at release):
  * shot_x, shot_y
  * dist_to_goal_center (m)
  * open_angle_to_goal (rad)
  * bearing_to_goal_center (rad)
- Kinematics (at release):
  * shooter_speed (m/s), shooter_acc (m/s²)
  * ball_speed (m/s), ball_acc (m/s²) if aligned
  * runup_angle_diff_to_goal (rad)
- Goalkeeper (if GK id known at release):
  * gk_x, gk_y, gk_dist_to_center, gk_lateral_offset, gk_line_depth
  * shooter_to_gk_distance (m)
- Defensive pressure (instant):
  * opp_within_1_5m, opp_within_3m
  * opp_in_shooting_cone_[15,30]deg (counts)
- Context:
  * is_goal (label)
  * home/away indicator for shooter team (if derivable)
  * game_clock/period if available (optional)

Notes:
- Field coords ass

In [3]:
# Cell 1 — Data load (reuses your connection & base frames where possible)



try:
    con
except NameError:
    import duckdb
    DB_PATH = "../data/mydb2024-25.duckdb"
    con = duckdb.connect(DB_PATH, read_only=False)
    print("Connected DuckDB:", DB_PATH)

# Positions (all sessions)
df_positions_all = con.execute("SELECT * FROM main.kinexon_positions").fetchdf()
if "ts" not in df_positions_all.columns:
    df_positions_all["ts"] = pd.to_datetime(df_positions_all["ts in ms"], unit="ms", utc=True, errors="coerce")

# Fixtures → session mapping (robust)
fx = con.execute("""
    SELECT fixtureId, COALESCE(session_id, NULL) AS session_id
    FROM main.fixtures
    WHERE session_id IS NOT NULL
""").fetchdf()
fixture_to_session = dict(zip(fx["fixtureId"], fx["session_id"]))

# Refined throw times (from your prior notebook output)
df_refined = con.execute("""
    SELECT fixtureId, eventId, refined_throw_ts, refined_throw_ts_ms,
           kinexon_matched_ts, kinexon_matched_ts_ms
    FROM main.kinexon_throwpoint_refined
""").fetchdf()
df_refined["refined_throw_ts"] = pd.to_datetime(df_refined["refined_throw_ts"], utc=True, errors="coerce")

# Match goals (labels via match_events)
df_goals = con.execute("""
    SELECT fixtureId, eventId, eventTime, eventType, personId, goalKeeperId,
           teamName, personName, goalkeeperName, session_id
    FROM main.match_events
    WHERE eventType = 'goal'
""").fetchdf()
df_goals["eventTime"] = pd.to_datetime(df_goals["eventTime"], utc=True, errors="coerce")

# Players (for league_id lookup)
players = con.execute("""
    SELECT personId, league_id, nameFullLatin AS person_name, teamName, entityId
    FROM main.players
""").fetchdf()

# Kinexon detected shots (goals + non-goals)
vw = con.execute("""
    SELECT fixture_id, session_id, kinexon_event_id, event_type, player_id,
           timestamp, timestamp_ms, success, shot_category, shot_position_x, shot_position_y,
           match_eventId, match_eventType, match_time_diff_ms
    FROM main.vw_kinexon_events_detected_with_matchid
    WHERE event_type = 'detected_shot_handball'
""").fetchdf()
vw["timestamp_ms"] = pd.to_numeric(vw["timestamp_ms"], errors="coerce")
vw["timestamp"] = pd.to_datetime(vw["timestamp"], utc=True, errors="coerce")

print(
    f"Loaded positions={len(df_positions_all):,}, refined={len(df_refined):,}, "
    f"goals={len(df_goals):,}, detected_shots={len(vw):,}"
)
display(vw.head(3))


Connected DuckDB: ../data/mydb2024-25.duckdb
Loaded positions=15,618,434, refined=20, goals=28,539, detected_shots=1,296


,fixture_id,session_id,kinexon_event_id,event_type,player_id,timestamp,timestamp_ms,success,shot_category,shot_position_x,shot_position_y,match_eventId,match_eventType,match_time_diff_ms
0,00c08679-4374-11ef-80bd-73cf0bc66b45,2658,15328703,detected_shot_handball,1480,2024-10-20 13:05:15+00:00,1729429515887,0,field,-17.055626,-7.126623,None,None,NaN
1,00c08679-4374-11ef-80bd-73cf0bc66b45,2658,15328789,detected_shot_handball,970,2024-10-20 13:05:51+00:00,1729429551049,1,field,-14.158737,-0.365233,09e0f481-8ee4-11ef-baff-6d233f85da92,goal,3111.0
2,00c08679-4374-11ef-80bd-73cf0bc66b45,2658,15328957,detected_shot_handball,1480,2024-10-20 13:06:27+00:00,1729429587462,1,field,-15.113567,-4.373566,2d298c41-8ee4-11ef-baff-6d233f85da92,goal,25894.0


In [4]:
# Cell 2 — Helper geometry & extraction utilities

from typing import Optional, Tuple

def _goal_posts_for_x(x: float) -> Tuple[Tuple[float,float], Tuple[float,float]]:
    """
    Choose the *nearest* goal along x to the shooter's x (robust to halves).
    If shooter x >= 0 → target goal at +GOAL_X_ABS; else at -GOAL_X_ABS.
    """
    gx = GOAL_X_ABS if x >= 0 else -GOAL_X_ABS
    left_post = (gx, -GOAL_HALF_WIDTH)
    right_post = (gx, +GOAL_HALF_WIDTH)
    return left_post, right_post

def _open_angle_to_goal(px: float, py: float) -> float:
    """Subtended angle between rays to left/right posts (radians)."""
    (gx_l, gy_l), (gx_r, gy_r) = _goal_posts_for_x(px)
    vL = np.array([gx_l - px, gy_l - py], dtype=float)
    vR = np.array([gx_r - px, gy_r - py], dtype=float)
    # angle between vL and vR
    num = np.dot(vL, vR)
    den = np.linalg.norm(vL) * np.linalg.norm(vR)
    den = den if den > 1e-9 else 1e-9
    cosang = np.clip(num / den, -1.0, 1.0)
    return float(np.pi - np.arccos(cosang))  # interior angle

def _bearing_to_goal_center(px: float, py: float) -> float:
    (gx_l, gy_l), (gx_r, gy_r) = _goal_posts_for_x(px)
    gx, gy = (gx_l + gx_r)/2.0, 0.0
    v = np.array([gx - px, gy - py], dtype=float)
    return float(np.arctan2(v[1], v[0]))     # [-pi, pi]

def _angle_diff(a: float, b: float) -> float:
    """Smallest signed difference a-b in radians."""
    d = (a - b + np.pi) % (2*np.pi) - np.pi
    return float(d)

def _row_time_ms(ts: Optional[pd.Timestamp], fallback_ms: Optional[int]) -> Optional[int]:
    if pd.notna(ts):
        return int(ts.value // 10**6)
    return int(fallback_ms) if pd.notna(fallback_ms) else None

def _asof_at(df: pd.DataFrame, ts: pd.Timestamp, tol_ms: int = NEAREST_TS_TOL_MS) -> pd.DataFrame:
    """
    Return all rows at the nearest timestamp to ts within tolerance (per 'ts' column).
    """
    if df.empty or pd.isna(ts): 
        return pd.DataFrame()
    # Find nearest frame by absolute delta
    df_ = df.copy()
    df_["absdt"] = (df_["ts"] - ts).abs()
    m = df_[df_["absdt"] <= pd.Timedelta(milliseconds=tol_ms)]
    if m.empty:
        return m
    min_abs = m["absdt"].min()
    return m[m["absdt"] == min_abs].drop(columns=["absdt"])

def _prev_frame_for_agent(df_agent: pd.DataFrame, ts: pd.Timestamp, ms_back: int = 50) -> Optional[pd.Series]:
    """Nearest previous sample for an agent before ts by ~ms_back."""
    if df_agent.empty or pd.isna(ts):
        return None
    target = ts - pd.Timedelta(milliseconds=ms_back)
    # pick nearest <= ts
    df_prev = df_agent[df_agent["ts"] <= ts].copy()
    if df_prev.empty:
        return None
    df_prev["absdt"] = (df_prev["ts"] - target).abs()
    return df_prev.sort_values("absdt").iloc[0].drop(labels=["absdt"], errors="ignore")


In [6]:
# Cell 3 — Candidate shot list (goals & non-goals) with release timestamps

# Prepare match goals with shooter/goalkeeper league ids
goals = df_goals.merge(
    players[["personId","league_id","teamName"]].rename(columns={"league_id":"shooter_league_id"}),
    on="personId", how="left"
).merge(
    players[["personId","league_id"]].rename(columns={"personId":"goalKeeperId","league_id":"goalkeeper_league_id"}),
    on="goalKeeperId", how="left"
)

# Attach refined throw times to goals
goals = goals.merge(
    df_refined[["fixtureId","eventId","refined_throw_ts","refined_throw_ts_ms"]],
    on=["fixtureId","eventId"], how="left"
)

# Build a common schema for detected shots (non-goals also)
shots = vw.rename(columns={
    "fixture_id":"fixtureId",
    "session_id":"session_id",
    "player_id":"kinexon_player_id",
    "timestamp":"detected_ts",
    "timestamp_ms":"detected_ms",
    "match_eventId":"eventId"
}).copy()

# Try to link Kinexon shooter to our players table (best-effort by Kinexon league 'player_id' if it matches)
shots["shooter_league_id"] = shots["kinexon_player_id"].astype("Int64")  # often same id set

# Merge session id for goals (in case missing)
goals["session_id"] = goals["session_id"].fillna(goals["fixtureId"].map(fixture_to_session))

# Union goals + detected shots (goals may also appear in shots; we won't dedupe aggressively here)
BASE_COLS = [
    "fixtureId","eventId","session_id",
    "shooter_league_id","goalkeeper_league_id",
    "detected_ts","detected_ms","shot_position_x","shot_position_y",
    "success","shot_category"
]
g_pack = goals.assign(
    detected_ts=pd.NaT, detected_ms=np.nan, success=1
)[BASE_COLS].copy()
s_pack = shots[BASE_COLS].copy()

candidates = pd.concat([g_pack, s_pack], ignore_index=True)

# Attach best release timestamp: refined (if eventId present) → detected_ms → NaN
candidates = candidates.merge(
    df_refined[["fixtureId","eventId","refined_throw_ts","refined_throw_ts_ms",
                "kinexon_matched_ts","kinexon_matched_ts_ms"]],
    on=["fixtureId","eventId"], how="left"
)

def choose_release_ts(row):
    if pd.notna(row.get("refined_throw_ts")):
        return row["refined_throw_ts"]
    if pd.notna(row.get("detected_ms")):
        return pd.to_datetime(int(row["detected_ms"]), unit="ms", utc=True, errors="coerce")
    if pd.notna(row.get("kinexon_matched_ts")):
        return row["kinexon_matched_ts"]
    return pd.NaT

candidates["release_ts"] = candidates.apply(choose_release_ts, axis=1)
candidates["label_is_goal"] = (candidates["success"].fillna(0) > 0).astype(int)

# Drop rows without session or timestamp
candidates = candidates[
    candidates["session_id"].notna() & candidates["release_ts"].notna()
].copy()

print("Candidate shots:", len(candidates))
display(candidates.head(5))


KeyError: "['shot_position_x', 'shot_position_y', 'shot_category'] not in index"

In [ ]:
# Cell 4 — Feature extraction at release

def compute_features_for_shot(row) -> Optional[pd.Series]:
    """
    Compute features at the instant of 'release_ts' using df_positions_all.
    """
    session_id = int(row["session_id"])
    ts_rel = pd.to_datetime(row["release_ts"], utc=True)
    shooter_id = row.get("shooter_league_id")
    gk_id = row.get("goalkeeper_league_id")

    # Slice nearest frame to release
    df_sess = df_positions_all[df_positions_all["session_id"] == session_id]
    frame = _asof_at(df_sess, ts_rel, tol_ms=NEAREST_TS_TOL_MS)
    if frame.empty:
        return None

    # Identify ball, shooter, GK, teams
    is_ball = frame["group id"] == 3
    ball = frame[is_ball].copy()

    shooter = frame[frame["league id"].astype(str) == str(shooter_id)] if pd.notna(shooter_id) else pd.DataFrame()
    if shooter.empty:
        # if shooter id missing, choose the nearest player to Kinexon detected position if present
        shooter = frame[(frame["group id"].isin([1,2]))].copy()

    gk = frame[frame["league id"].astype(str) == str(gk_id)] if pd.notna(gk_id) else pd.DataFrame()

    # If multiple rows per role, pick first
    shooter_row = shooter.iloc[0] if not shooter.empty else None
    ball_row    = ball.iloc[0]    if not ball.empty else None
    gk_row      = gk.iloc[0]      if not gk.empty else None

    # Shooter kinematics & position
    if shooter_row is None:
        return None

    sx, sy = float(shooter_row["x in m"]), float(shooter_row["y in m"])
    ss, sa = float(shooter_row.get("speed in m/s", np.nan)), float(shooter_row.get("acceleration in m/s2", np.nan))

    # Previous sample for run-up angle
    df_agent = df_sess[df_sess["league id"].astype(str) == str(shooter_id)].sort_values("ts")
    prev = _prev_frame_for_agent(df_agent, ts_rel, ms_back=50)
    if prev is not None:
        vx, vy = sx - float(prev["x in m"]), sy - float(prev["y in m"])
        move_bearing = float(np.arctan2(vy, vx)) if (abs(vx)+abs(vy)) > 1e-6 else np.nan
    else:
        move_bearing = np.nan

    # Geometry to goal
    dist_to_center = float(np.hypot((_goal_posts_for_x(sx)[0][0] + _goal_posts_for_x(sx)[1][0])/2 - sx, 0.0 - sy))
    open_angle = _open_angle_to_goal(sx, sy)
    goal_bearing = _bearing_to_goal_center(sx, sy)
    runup_angle_diff = abs(_angle_diff(move_bearing, goal_bearing)) if not np.isnan(move_bearing) else np.nan

    # Ball kinematics (if available)
    if ball_row is not None:
        bs = float(ball_row.get("speed in m/s", np.nan))
        ba = float(ball_row.get("acceleration in m/s2", np.nan))
        bx, by = float(ball_row["x in m"]), float(ball_row["y in m"])
        dist_pb = float(np.hypot(bx - sx, by - sy))
    else:
        bs = ba = dist_pb = np.nan

    # Goalkeeper features
    if gk_row is not None:
        gx, gy = float(gk_row["x in m"]), float(gk_row["y in m"])
        gk_dist_center = float(np.hypot((_goal_posts_for_x(sx)[0][0] + _goal_posts_for_x(sx)[1][0])/2 - gx, 0.0 - gy))
        gk_lateral_offset = float(gy)  # relative to goal center y=0
        gk_line_depth = float((_goal_posts_for_x(sx)[0][0]) - gx)  # +: in front of goal line (toward court)
        shooter_to_gk = float(np.hypot(gx - sx, gy - sy))
    else:
        gx = gy = gk_dist_center = gk_lateral_offset = gk_line_depth = shooter_to_gk = np.nan

    # Defensive pressure counts
    if "group id" in frame.columns and not shooter.empty:
        shooter_gid = int(shooter_row["group id"]) if pd.notna(shooter_row["group id"]) else None
        field_players = frame[frame["group id"].isin([1,2])].copy()
        # opponents: different group id than shooter (ignore NaN guards)
        if shooter_gid in [1,2]:
            opp = field_players[field_players["group id"] != shooter_gid]
        else:
            opp = field_players
        dx = opp["x in m"].to_numpy() - sx
        dy = opp["y in m"].to_numpy() - sy
        d = np.hypot(dx, dy)
        # bearings of opponents relative to shooter and goal
        opp_bearing = np.arctan2(dy, dx)
        delta_to_goal_bearing = np.abs((opp_bearing - goal_bearing + np.pi) % (2*np.pi) - np.pi)

        opp_r1 = int(np.sum(d <= PRESSURE_R1))
        opp_r2 = int(np.sum(d <= PRESSURE_R2))
        opp_cone15 = int(np.sum((delta_to_goal_bearing <= CONE_15)))
        opp_cone30 = int(np.sum((delta_to_goal_bearing <= CONE_30)))
    else:
        opp_r1 = opp_r2 = opp_cone15 = opp_cone30 = 0

    return pd.Series(dict(
        fixtureId=row["fixtureId"],
        eventId=row.get("eventId"),
        session_id=session_id,
        release_ts=ts_rel,
        shooter_league_id=shooter_id,
        goalkeeper_league_id=gk_id,
        # geometry
        shot_x=sx, shot_y=sy,
        dist_to_goal_center=dist_to_center,
        open_angle_to_goal=open_angle,
        bearing_to_goal_center=goal_bearing,
        # kinematics
        shooter_speed=ss, shooter_acc=sa,
        ball_speed=bs, ball_acc=ba, dist_player_ball=dist_pb,
        runup_angle_diff=runup_angle_diff,
        # goalkeeper
        gk_x=gx, gk_y=gy, gk_dist_to_center=gk_dist_center,
        gk_lateral_offset=gk_lateral_offset, gk_line_depth=gk_line_depth,
        shooter_to_gk_distance=shooter_to_gk,
        # pressure
        opp_within_1_5m=opp_r1, opp_within_3m=opp_r2,
        opp_in_cone_15deg=opp_cone15, opp_in_cone_30deg=opp_cone30,
        # label & provenance
        is_goal=int(row["label_is_goal"]),
        source_has_refined=int(pd.notna(row.get("refined_throw_ts"))),
    ))

# Run feature extraction
features = []
for i, r in candidates.iterrows():
    f = compute_features_for_shot(r)
    if f is not None:
        features.append(f)
    if (i+1) % 200 == 0:
        print(f"Processed {i+1}/{len(candidates)}")

df_xg = pd.DataFrame(features).sort_values("release_ts").reset_index(drop=True)
print("Built features:", len(df_xg))
display(df_xg.head(10))


In [ ]:
# Cell 5 — Basic QC and descriptive stats

def _describe(df, cols):
    out = {}
    for c in cols:
        s = pd.to_numeric(df[c], errors="coerce")
        out[c] = dict(count=int(s.notna().sum()),
                      mean=float(s.mean(skipna=True)) if s.notna().any() else np.nan,
                      std=float(s.std(skipna=True)) if s.notna().any() else np.nan,
                      min=float(s.min(skipna=True)) if s.notna().any() else np.nan,
                      p50=float(s.quantile(0.5)) if s.notna().any() else np.nan,
                      max=float(s.max(skipna=True)) if s.notna().any() else np.nan)
    return pd.DataFrame(out).T

qc_cols = [
    "dist_to_goal_center","open_angle_to_goal","shooter_speed","ball_speed",
    "runup_angle_diff","opp_within_1_5m","opp_within_3m","opp_in_cone_15deg","opp_in_cone_30deg",
    "gk_dist_to_center","shooter_to_gk_distance"
]
qc = _describe(df_xg, qc_cols)
display(qc)
print("Goal rate:", df_xg["is_goal"].mean().round(3), "(n=", len(df_xg), ")")


In [ ]:
# Cell 6 — Persist feature table to DuckDB (and optional CSV)

TABLE = "main.xg_shot_features"
con.execute(f"DROP TABLE IF EXISTS {TABLE}")
con.register("df_xg", df_xg)
con.execute(f"CREATE TABLE {TABLE} AS SELECT * FROM df_xg")
con.unregister("df_xg")

rows = con.execute(f"SELECT COUNT(*) FROM {TABLE}").fetchone()[0]
print(f"✓ Wrote {rows} rows to {TABLE}")

# Optional: also export to CSV for modeling outside DuckDB
SAVE_CSV = True
if SAVE_CSV:
    OUT_PATH = "../data/exports/xg_shot_features.csv"
    import os, pathlib
    pathlib.Path(os.path.dirname(OUT_PATH)).mkdir(parents=True, exist_ok=True)
    df_xg.to_csv(OUT_PATH, index=False)
    print("CSV saved at:", OUT_PATH)


In [ ]:
# Cell 7 — (Optional) quick train/val split preview inside DuckDB

con.execute("""
    CREATE OR REPLACE TABLE main.xg_shot_features_split AS
    WITH base AS (
        SELECT *,
               (hash(fixtureId) % 10) AS fold10
        FROM main.xg_shot_features
    )
    SELECT * FROM base
""")
print("Prepared fold10 split in main.xg_shot_features_split (0..9).")

display(con.execute("""
    SELECT fold10, COUNT(*) AS n, AVG(is_goal)::DOUBLE AS goal_rate
    FROM main.xg_shot_features_split
    GROUP BY 1 ORDER BY 1
""").fetchdf())
